## 실전 예제: 번역 품질 개선기

In [1]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

class TranslationState(TypedDict):
    original: str
    translated: str
    review: str
    final_output: str

model = init_chat_model("openai:gpt-4o-mini")

def translate(state: TranslationState) -> dict:
    """1단계: 영어 → 한국어 번역"""
    msg = model.invoke(
        f"다음 영어 문장을 한국어로 번역하세요. 번역문만 출력하세요.\n\n{state['original']}"
    )
    print(f"[translate] {msg.content}")
    return {"translated": msg.content}

def review(state: TranslationState) -> dict:
    """2단계: 번역 품질 검수"""
    msg = model.invoke(
        f"다음 번역의 품질을 한 줄로 평가하세요.\n"
        f"원문: {state['original']}\n번역: {state['translated']}"
    )
    print(f"[review] {msg.content}")
    return {"review": msg.content}

def finalize(state: TranslationState) -> dict:
    """3단계: 최종 출력 생성"""
    final = f"번역: {state['translated']}\n검수: {state['review']}"
    print(f"[finalize] 완료")
    return {"final_output": final}

# 그래프 구성
builder = StateGraph(TranslationState)
builder.add_node("translate", translate)
builder.add_node("review", review)
builder.add_node("finalize", finalize)
builder.add_edge(START, "translate")
builder.add_edge("translate", "review")
builder.add_edge("review", "finalize")
builder.add_edge("finalize", END)

# 체크포인터와 함께 컴파일
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


## 초기실행

In [2]:
config = {"configurable": {"thread_id": "translation_1"}}

result = graph.invoke(
    {
        "original": "The early bird catches the worm.",
        "translated": "",
        "review": "",
        "final_output": "",
    },
    config=config,
)

print("\n=== 초기 결과 ===")
print(result["final_output"])


[translate] 일찍 일어나는 새가 벌레를 잡는다.
[review] 번역이 원문의 의미와 뉘앙스를 잘 전달하고 있어 품질이 높습니다.
[finalize] 완료

=== 초기 결과 ===
번역: 일찍 일어나는 새가 벌레를 잡는다.
검수: 번역이 원문의 의미와 뉘앙스를 잘 전달하고 있어 품질이 높습니다.


## 체크포인트 탐색

In [3]:
history = list(graph.get_state_history(config))

print(f"총 {len(history)}개의 체크포인트\n")

for i, snapshot in enumerate(history):
    step = snapshot.metadata.get("step", "N/A")
    next_nodes = snapshot.next
    cp_id = snapshot.config["configurable"]["checkpoint_id"]

    print(f"[체크포인트 {i}] Step {step}")
    print(f"  다음 노드: {next_nodes}")
    print(f"  ID: {cp_id[:20]}...")

    # 상태 요약
    vals = snapshot.values
    if vals.get("translated"):
        print(f"  번역: {vals['translated'][:50]}...")
    print()


총 5개의 체크포인트

[체크포인트 0] Step 3
  다음 노드: ()
  ID: 1f133b74-c3d6-63fb-8...
  번역: 일찍 일어나는 새가 벌레를 잡는다....

[체크포인트 1] Step 2
  다음 노드: ('finalize',)
  ID: 1f133b74-c3d3-6e0b-8...
  번역: 일찍 일어나는 새가 벌레를 잡는다....

[체크포인트 2] Step 1
  다음 노드: ('review',)
  ID: 1f133b74-ba7f-6d6a-8...
  번역: 일찍 일어나는 새가 벌레를 잡는다....

[체크포인트 3] Step 0
  다음 노드: ('translate',)
  ID: 1f133b74-a828-6d4d-8...

[체크포인트 4] Step -1
  다음 노드: ('__start__',)
  ID: 1f133b74-a826-663d-b...

